<a href="https://colab.research.google.com/github/nevita275/ML_20_Nevita/blob/main/JS09/JS09-Tugas1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NAMA: NEVITA TRIYA YULIANA
# KELAS: TI-3F
# ABSEN: 20 <br><br>


# Approximate Nearest Neighbors (ANN)

# TUGAS LAB 1 <br> <br>

## Langkah 1 - Install Library

In [1]:
!pip install annoy faiss-cpu hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 5.9 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 6.4 MB/s  0:00:02
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520032 sha256=59a1c1c071a7dea817cc3ea3454d1a1fa9193a7923f06a1234b536818283735f
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679798 sha256=d069eac346c74d36382d29ed99312f2a0b7dd9ea15468a69f15db1e0b6709838
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e

## Langkah 2 - Import dan parameter

In [2]:
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import faiss
import hnswlib
from annoy import AnnoyIndex

K = 10                 # jumlah tetangga
NQ = 100               # jumlah query untuk dirata-rata
N_BESAR = 1_000_000    # ganti 100_000 untuk uji coba cepat dulu
CONFIGS = [(1000, 2), (1000, 5), (N_BESAR, 2), (N_BESAR, 5)]
METRICS = ["euclidean", "cosine"]

## Langkah 3 - Fungsi bantu dan fungsi benchmark

In [3]:
def normalize(A):
    A = A.copy()
    faiss.normalize_L2(A)
    return A

def ground_truth(X, Q, metric):
    """Tetangga sebenarnya (exact) sebagai pembanding recall."""
    d = X.shape[1]
    if metric == "euclidean":
        idx = faiss.IndexFlatL2(d)
        idx.add(X)
        _, I = idx.search(Q, K)
    else:
        idx = faiss.IndexFlatIP(d)
        idx.add(normalize(X))
        _, I = idx.search(normalize(Q), K)
    return I

def recall(I_pred, I_gt):
    return np.mean([len(set(p) & set(g)) / K for p, g in zip(I_pred, I_gt)])

def run_annoy(X, Q, metric, gt):
    d = X.shape[1]
    idx = AnnoyIndex(d, "euclidean" if metric == "euclidean" else "angular")
    t0 = time.time()
    for i in range(len(X)):
        idx.add_item(i, X[i])
    idx.build(10)
    build = time.time() - t0

    t0 = time.time()
    I = [idx.get_nns_by_vector(q, K, search_k=1000) for q in Q]
    qt = (time.time() - t0) * 1000 / len(Q)
    return build, qt, recall(I, gt)

def run_faiss_flat(X, Q, metric, gt):
    """FAISS exact (seperti Lab 4)."""
    d = X.shape[1]
    if metric == "euclidean":
        Xi, Qi = X, Q
        idx = faiss.IndexFlatL2(d)
    else:
        Xi, Qi = normalize(X), normalize(Q)
        idx = faiss.IndexFlatIP(d)

    t0 = time.time()
    idx.add(Xi)
    build = time.time() - t0

    t0 = time.time()
    I = [idx.search(Qi[i:i+1], K)[1][0] for i in range(len(Qi))]
    qt = (time.time() - t0) * 1000 / len(Qi)
    return build, qt, recall(I, gt)

def run_faiss_ivf(X, Q, metric, gt):
    """FAISS aproksimasi (IVFFlat)."""
    d, n = X.shape[1], len(X)
    nlist = 16 if n <= 10000 else 1000
    if metric == "euclidean":
        Xi, Qi = X, Q
        quant = faiss.IndexFlatL2(d)
        idx = faiss.IndexIVFFlat(quant, d, nlist, faiss.METRIC_L2)
    else:
        Xi, Qi = normalize(X), normalize(Q)
        quant = faiss.IndexFlatIP(d)
        idx = faiss.IndexIVFFlat(quant, d, nlist, faiss.METRIC_INNER_PRODUCT)

    t0 = time.time()
    idx.train(Xi)
    idx.add(Xi)
    build = time.time() - t0

    idx.nprobe = 10
    t0 = time.time()
    I = [idx.search(Qi[i:i+1], K)[1][0] for i in range(len(Qi))]
    qt = (time.time() - t0) * 1000 / len(Qi)
    return build, qt, recall(I, gt)

def run_hnsw(X, Q, metric, gt):
    d, n = X.shape[1], len(X)
    p = hnswlib.Index(space="l2" if metric == "euclidean" else "cosine", dim=d)
    t0 = time.time()
    p.init_index(max_elements=n, ef_construction=100, M=16)
    p.add_items(X)
    build = time.time() - t0

    p.set_ef(50)
    t0 = time.time()
    I = [p.knn_query(Q[i:i+1], k=K)[0][0] for i in range(len(Q))]
    qt = (time.time() - t0) * 1000 / len(Q)
    return build, qt, recall(I, gt)

## Langkah 4 - Jalankan semua percobaan

In [4]:
np.random.seed(42)
rows = []

for n, d in CONFIGS:
    X = np.random.random((n, d)).astype("float32")
    Q = np.random.random((NQ, d)).astype("float32")
    for metric in METRICS:
        gt = ground_truth(X, Q, metric)
        for name, fn in (("ANNOY", run_annoy),
                         ("FAISS-Flat", run_faiss_flat),
                         ("FAISS-IVF", run_faiss_ivf),
                         ("HNSW", run_hnsw)):
            b, q, r = fn(X, Q, metric, gt)
            rows.append(dict(Data=f"{n:,}/{d}D", Metric=metric, Algo=name,
                             Build_s=b, Query_ms=q, Recall=r))
            print(f"{n:,}/{d}D | {metric:9s} | {name:10s} | build={b:8.3f}s | query={q:8.4f}ms | recall={r:.3f}")

df = pd.DataFrame(rows)

1,000/2D | euclidean | ANNOY      | build=   0.024s | query=  0.1432ms | recall=1.000
1,000/2D | euclidean | FAISS-Flat | build=   0.000s | query=  0.0195ms | recall=1.000
1,000/2D | euclidean | FAISS-IVF  | build=   0.001s | query=  0.0228ms | recall=1.000
1,000/2D | euclidean | HNSW       | build=   0.029s | query=  0.0178ms | recall=1.000
1,000/2D | cosine    | ANNOY      | build=   0.036s | query=  0.1278ms | recall=1.000
1,000/2D | cosine    | FAISS-Flat | build=   0.000s | query=  0.0154ms | recall=1.000
1,000/2D | cosine    | FAISS-IVF  | build=   0.001s | query=  0.0230ms | recall=1.000
1,000/2D | cosine    | HNSW       | build=   0.023s | query=  0.0152ms | recall=0.999
1,000/5D | euclidean | ANNOY      | build=   0.017s | query=  0.1235ms | recall=1.000
1,000/5D | euclidean | FAISS-Flat | build=   0.000s | query=  0.0156ms | recall=1.000
1,000/5D | euclidean | FAISS-IVF  | build=   0.002s | query=  0.0227ms | recall=1.000
1,000/5D | euclidean | HNSW       | build=   0.033s | 

## Tabel Hasil

In [5]:
KOLOM_ALGO = ["ANNOY", "FAISS-Flat", "FAISS-IVF", "HNSW"]

df["Hasil"] = df.apply(
    lambda r: f"B {r.Build_s:.2f}s | Q {r.Query_ms:.3f}ms | R {r.Recall:.2f}", axis=1)

print("=== TABEL RINGKAS (B = build, Q = query per query, R = recall@10) ===")
ringkas = df.pivot_table(index=["Data", "Metric"], columns="Algo", values="Hasil",
                         aggfunc="first", sort=False)[KOLOM_ALGO]
display(ringkas)

for kolom, judul in (("Build_s", "Waktu build (detik)"),
                     ("Query_ms", "Waktu query (ms/query)"),
                     ("Recall", "Recall@10")):
    t = df.pivot_table(index=["Data", "Metric"], columns="Algo", values=kolom,
                       aggfunc="first", sort=False)[KOLOM_ALGO]
    print("\n" + judul)
    display(t.round(4))

=== TABEL RINGKAS (B = build, Q = query per query, R = recall@10) ===


Algo                                             ANNOY  \
Data         Metric                                      
1,000/2D     euclidean    B 0.02s | Q 0.143ms | R 1.00   
             cosine       B 0.04s | Q 0.128ms | R 1.00   
1,000/5D     euclidean    B 0.02s | Q 0.124ms | R 1.00   
             cosine       B 0.02s | Q 0.133ms | R 1.00   
1,000,000/2D euclidean   B 30.27s | Q 0.226ms | R 1.00   
             cosine     B 104.67s | Q 0.220ms | R 0.05   
1,000,000/5D euclidean   B 23.90s | Q 0.436ms | R 1.00   
             cosine      B 33.99s | Q 0.251ms | R 1.00   

Algo                                       FAISS-Flat  \
Data         Metric                                     
1,000/2D     euclidean   B 0.00s | Q 0.019ms | R 1.00   
             cosine      B 0.00s | Q 0.015ms | R 1.00   
1,000/5D     euclidean   B 0.00s | Q 0.016ms | R 1.00   
             cosine      B 0.00s | Q 0.016ms | R 1.00   
1,000,000/2D euclidean   B 0.00s | Q 8.219ms | R 0.99   
             cosine      B 0.00s | Q 2.245ms | R 0.99   
1,000,000/5D euclidean  B 0.03s | Q 18.769ms | R 1.00   
             cosine      B 0.02s | Q 3.414ms | R 1.00   

Algo                                       FAISS-IVF  \
Data         Metric                                    
1,000/2D     euclidean  B 0.00s | Q 0.023ms | R 1.00   
             cosine     B 0.00s | Q 0.023ms | R 1.00   
1,000/5D     euclidean  B 0.00s | Q 0.023ms | R 1.00   
             cosine     B 0.00s | Q 0.025ms | R 1.00   
1,000,000/2D euclidean  B 1.29s | Q 0.156ms | R 1.00   
             cosine     B 4.25s | Q 0.325ms | R 0.95   
1,000,000/5D euclidean  B 1.21s | Q 0.132ms | R 1.00   
             cosine     B 6.74s | Q 0.154ms | R 1.00   

Algo                                              HNSW  
Data         Metric                                     
1,000/2D     euclidean    B 0.03s | Q 0.018ms | R 1.00  
             cosine       B 0.02s | Q 0.015ms | R 1.00  
1,000/5D     euclidean    B 0.03s | Q 0.025ms | R 1.00  
             cosine       B 0.03s | Q 0.024ms | R 1.00  
1,000,000/2D euclidean   B 69.19s | Q 0.049ms | R 1.00  
             cosine      B 99.39s | Q 0.069ms | R 0.27  
1,000,000/5D euclidean  B 105.66s | Q 0.068ms | R 1.00  
             cosine      B 97.82s | Q 0.071ms | R 1.00


Waktu build (detik)


Algo                       ANNOY  FAISS-Flat  FAISS-IVF      HNSW
Data         Metric                                              
1,000/2D     euclidean    0.0241      0.0000     0.0014    0.0292
             cosine       0.0358      0.0000     0.0014    0.0229
1,000/5D     euclidean    0.0172      0.0000     0.0015    0.0333
             cosine       0.0238      0.0000     0.0016    0.0326
1,000,000/2D euclidean   30.2696      0.0039     1.2877   69.1915
             cosine     104.6738      0.0030     4.2534   99.3928
1,000,000/5D euclidean   23.9030      0.0250     1.2073  105.6572
             cosine      33.9876      0.0152     6.7395   97.8173


Waktu query (ms/query)


Algo                     ANNOY  FAISS-Flat  FAISS-IVF    HNSW
Data         Metric                                          
1,000/2D     euclidean  0.1432      0.0195     0.0228  0.0178
             cosine     0.1278      0.0154     0.0230  0.0152
1,000/5D     euclidean  0.1235      0.0156     0.0227  0.0249
             cosine     0.1326      0.0162     0.0248  0.0240
1,000,000/2D euclidean  0.2260      8.2194     0.1559  0.0492
             cosine     0.2197      2.2446     0.3254  0.0692
1,000,000/5D euclidean  0.4365     18.7690     0.1322  0.0679
             cosine     0.2507      3.4141     0.1537  0.0711


Recall@10


Algo                    ANNOY  FAISS-Flat  FAISS-IVF   HNSW
Data         Metric                                        
1,000/2D     euclidean   1.00       1.000      1.000  1.000
             cosine      1.00       1.000      1.000  0.999
1,000/5D     euclidean   1.00       1.000      1.000  1.000
             cosine      1.00       1.000      1.000  1.000
1,000,000/2D euclidean   1.00       0.987      1.000  1.000
             cosine      0.05       0.993      0.948  0.270
1,000,000/5D euclidean   1.00       1.000      1.000  1.000
             cosine      1.00       1.000      1.000  1.000

In [7]:
from IPython.display import HTML, display

NAMA_ALGO = ["ANNOY", "FAISS-Flat", "FAISS-IVF", "HNSW"]

def isi_sel(r):
    return f"Build: {r.Build_s:.2f} s<br>Query: {r.Query_ms:.3f} ms<br>Recall: {r.Recall:.2f}"

df["Sel"] = df.apply(isi_sel, axis=1)
df["Konfigurasi"] = (df["Data"].str.replace(",", "", regex=False)
                     + " (" + df["Metric"].str.capitalize() + ")")

tabel = df.pivot_table(index="Konfigurasi", columns="Algo", values="Sel",
                       aggfunc="first", sort=False)[NAMA_ALGO]
tabel.index.name = "Jumlah Data/Dimensi"
tabel.columns.name = None

# Tampilan tabel
gaya = """
<style>
.tbl-ann {border-collapse:collapse; font-size:14px; background:#fff; color:#111;}
.tbl-ann th, .tbl-ann td {border:1px solid #d0d7de; padding:8px 12px; text-align:left; vertical-align:top;}
.tbl-ann th {background:#f3f4f8;}
</style>
"""
display(HTML(gaya + tabel.to_html(escape=False, classes="tbl-ann", border=0)))



,ANNOY,FAISS-Flat,FAISS-IVF,HNSW
Jumlah Data/Dimensi,,,,
1000/2D (Euclidean),Build: 0.02 sQuery: 0.143 msRecall: 1.00,Build: 0.00 sQuery: 0.019 msRecall: 1.00,Build: 0.00 sQuery: 0.023 msRecall: 1.00,Build: 0.03 sQuery: 0.018 msRecall: 1.00
1000/2D (Cosine),Build: 0.04 sQuery: 0.128 msRecall: 1.00,Build: 0.00 sQuery: 0.015 msRecall: 1.00,Build: 0.00 sQuery: 0.023 msRecall: 1.00,Build: 0.02 sQuery: 0.015 msRecall: 1.00
1000/5D (Euclidean),Build: 0.02 sQuery: 0.124 msRecall: 1.00,Build: 0.00 sQuery: 0.016 msRecall: 1.00,Build: 0.00 sQuery: 0.023 msRecall: 1.00,Build: 0.03 sQuery: 0.025 msRecall: 1.00
1000/5D (Cosine),Build: 0.02 sQuery: 0.133 msRecall: 1.00,Build: 0.00 sQuery: 0.016 msRecall: 1.00,Build: 0.00 sQuery: 0.025 msRecall: 1.00,Build: 0.03 sQuery: 0.024 msRecall: 1.00
1000000/2D (Euclidean),Build: 30.27 sQuery: 0.226 msRecall: 1.00,Build: 0.00 sQuery: 8.219 msRecall: 0.99,Build: 1.29 sQuery: 0.156 msRecall: 1.00,Build: 69.19 sQuery: 0.049 msRecall: 1.00
1000000/2D (Cosine),Build: 104.67 sQuery: 0.220 msRecall: 0.05,Build: 0.00 sQuery: 2.245 msRecall: 0.99,Build: 4.25 sQuery: 0.325 msRecall: 0.95,Build: 99.39 sQuery: 0.069 msRecall: 0.27
1000000/5D (Euclidean),Build: 23.90 sQuery: 0.436 msRecall: 1.00,Build: 0.03 sQuery: 18.769 msRecall: 1.00,Build: 1.21 sQuery: 0.132 msRecall: 1.00,Build: 105.66 sQuery: 0.068 msRecall: 1.00
1000000/5D (Cosine),Build: 33.99 sQuery: 0.251 msRecall: 1.00,Build: 0.02 sQuery: 3.414 msRecall: 1.00,Build: 6.74 sQuery: 0.154 msRecall: 1.00,Build: 97.82 sQuery: 0.071 msRecall: 1.00


## Analisis

Percobaan dilakukan pada data acak berukuran 1.000 dan 1.000.000 dengan dimensi
2D dan 5D, memakai metric Euclidean dan Cosine. Parameter: ANNOY 10 pohon
(search_k=1000), FAISS-Flat (exact), FAISS-IVF (IVFFlat, nprobe=10), dan HNSW
(M=16, ef_construction=100, ef=50). Waktu query adalah rata-rata dari 100 query
yang dijalankan satu per satu, dan recall@10 dihitung terhadap pencarian exact.

**Data kecil (1.000 titik).** Keempat metode memiliki recall sekitar 1,0 dan
waktu yang sangat kecil. Query FAISS-Flat, FAISS-IVF, dan HNSW berada di kisaran
0,015-0,025 ms, sedangkan ANNOY sekitar 0,12-0,14 ms, sehingga ANNOY paling
lambat walaupun selisih absolutnya kecil. Pada skala ini pencarian exact sudah
cukup, dan keunggulan metode aproksimasi belum terlihat.

**Data besar (1.000.000 titik).** Perbedaannya jelas. FAISS-Flat (exact) hampir
tidak memerlukan waktu build tetapi query-nya paling lambat, yaitu 2-19 ms per
query. HNSW memiliki query tercepat (0,049-0,071 ms), sekitar 270 kali lebih
cepat dari FAISS-Flat pada 5D Euclidean (0,068 ms vs 18,769 ms), tetapi waktu
build-nya paling lama (69-106 detik). FAISS-IVF menjadi kompromi: build hanya
1,2-6,7 detik dengan query 0,13-0,33 ms. ANNOY memiliki query 0,22-0,44 ms dan
waktu build yang bervariasi besar (24-105 detik), paling lama pada Cosine 2D.

**Pengaruh metric dan dimensi.** Untuk Euclidean, recall seluruh metode ANN adalah
1,00 pada 2D maupun 5D. Untuk Cosine pada 5D, semua metode juga mencapai recall
1,00. Pengecualiannya adalah Cosine pada 2D dengan 1.000.000 data: recall ANNOY
hanya 0,05, HNSW 0,27, dan FAISS-IVF 0,948. Pada 2D kemiripan cosine hanya
ditentukan oleh sudut, sehingga banyak titik memiliki selisih kemiripan yang
sangat kecil dan sulit dibedakan oleh pencarian aproksimasi. FAISS-Flat pada 2D
menunjukkan recall 0,987-0,993 (seharusnya 1,00), yang kemungkinan disebabkan
oleh pembulatan float32 pada titik berjarak hampir sama.

**Keterbatasan.** Datanya acak seragam, parameter setiap metode tidak di-tuning,
dan waktu diukur satu kali per konfigurasi sehingga dipengaruhi kondisi mesin
Colab. Hasil sebaiknya dibaca sebagai pola umum, bukan angka mutlak.

**Kesimpulan.** Exact search cukup untuk data kecil. Pada data besar, HNSW cocok
bila kecepatan query paling penting dan waktu build bisa diterima, FAISS-IVF
cocok bila build harus cepat dengan query yang tetap cepat, dan ANNOY cocok
sebagai alternatif dengan performa query menengah. Pemilihan metode juga perlu
mempertimbangkan metric dan dimensi data, karena recall Cosine pada 2D turun
tajam pada ANNOY dan HNSW.